[FACTS Grounding](https://kaggle.com/facts-leaderboard) is a novel benchmark from Google DeepMind and Google Research designed to evaluate the factual accuracy and grounding of AI models.

**As a primer, please check out this [Examples Section](https://kaggle.com/facts-leaderboard/examples) before running this notebook!**

This notebook will walk you through:

1) **Generating responses** for examples

2) **Evaluating responses** that were generated

3) **Ensembling evaluations** to produce a score

In [1]:
import kaggle_benchmarks as kbench

import re
import json

In [2]:
import pandas as pd

examples = pd.read_csv("/kaggle/input/FACTS-grounding-examples/examples.csv")
examples

,system_instruction,user_request,context_document,full_prompt,domain,type,high_level_type
0,"{instruction}\n ==========\n In your answer, r...","I'm middle-aged, never smoked, had my ears blo...",high blood pressure \n People who have consist...,"{instruction}\n ==========\n In your answer, r...",Medical,Effect Analysis,Q&A
1,You formulate answers based solely on the mate...,Can you list all the knife brands that sell kn...,Knife sharpening angles\nManufacturer´s recomm...,You formulate answers based solely on the mate...,Retail/Product,Fact Finding,Q&A
2,Provide your response in a professional and fo...,What are some tips on saving money?,Money Management Tips: 55 Ways to Save Money\n...,Provide your response in a professional and fo...,Financial,Find & Summarize,Text Transformation
3,You must only use the context to answer the qu...,What are all the contexts when it is right fo...,Description of the disease: Leptospirosis is a...,System instruction: You must only use the cont...,Medical,Fact Finding,Q&A
4,Do not use any information other than that con...,can you summarise all the important informatio...,This Regulation provides for full harmonisatio...,system instruction: [Do not use any informatio...,Legal,Summarize,Text Transformation
...,...,...,...,...,...,...,...
852,"Using only the provided text, answer all follo...","What is the meaning of ""forward-looking statem...",JOANN Receives Court Approval for Prepackaged ...,"Using only the provided text, answer all follo...",Legal,Explanation/Definition,Q&A
853,Provide your response in a professional and fo...,What are all of the different landline feature...,CentraNet\nCustoPAK\n®\n®\nUSER GUI DE\nTeleph...,Provide your response in a professional and fo...,Internet/Technology,Fact Finding,Q&A
854,"""================\n <TEXT PASSAGE>\n =======\n...",Given the recent advancements of mRNA technolo...,"Annually, seasonal influenza is responsible fo...","""================\n <TEXT PASSAGE>\n =======\n...",Medical,Find & Summarize,Text Transformation
855,"For this task, you must only answer using the ...",Summarise the court findings of the Silkroad case,"Silk Road\nIn 2013, federal authorities shut d...",Summarise the court findings of the Silkroad c...,Legal,Summarize,Text Transformation


In [3]:
JUDGES = [
    "google/gemini-2.5-flash",
    "openai/gpt-5-2025-08-07",
]

In [4]:
# Limit to the first 5 examples
examples = examples.head(5)

## Quality Prompts and Helper Functions

In [5]:
def extract_instruction_json(ans: str) -> dict:
    pattern = re.compile(
        r'{\s*'
        r'"Instruction Following"\s*:\s*'
        r'"(No Issues|Minor Issue\(s\)|Major Issue\(s\))"\s*'
        r'}'
    )

    match = pattern.search(ans)

    if match:
        json_string = match.group(0)
        try:
            # Once the valid JSON string is found, parse it
            return json.loads(json_string)
        except Exception as e:
            # This is unlikely to happen if the regex matches, but it's good practice
            print("!!Bad Json!!")
            print(e)
            print(ans)
            return {"Instruction Following": "Invalid"}

    return {"Instruction Following": "Invalid"}



def _evaluate_quality_no_context(
    user_request: str, response_a: str, response_b: str, judge: str
):
    """Evaluates response_a for quality using response_b as a reference."""
    ineligible_responses_filter_no_context_prompt = """Your mission is to judge the response from an AI model, the *test* response, calibrating your judgement using a *baseline* response.
Please use the following rubric criteria to judge the responses:

<START OF RUBRICS>
Your task is to analyze the test response based on the criterion of "Instruction Following". Start your analysis with "Analysis".

**Instruction Following**
Please first list the instructions in the user query.
In general, an instruction is VERY important if it is specifically asked for in the prompt and deviates from the norm. Please highlight such specific keywords.
You should also derive the task type from the user query and include the task-specific implied instructions.
Sometimes, no instruction is available in the user query.
It is your job to infer if the instruction is to autocomplete the user query or is asking the LLM for follow-ups.
After listing the instructions, you should rank them in order of importance.
After that, INDEPENDENTLY check if the test response and the baseline response meet each of the instructions.
You should itemize, for each instruction, whether the response meets, partially meets, or does not meet the requirement, using reasoning.
You should start reasoning first before reaching a conclusion about whether the response satisfies the requirement.
Citing examples while reasoning is preferred.

Important: Only select `Major Issue(s)` for issues that are extremely egregious and critical to the response, such that they give little to no value to the user.

Note: Both the `baseline` and `test` responses were made based on a given context document **which was redacted** in your analysis. You do not have access to the context document which is likely referenced in the user request and model responses (both baseline and test) - assume that it contains the information referenced adequately.

Reflect on your answer and consider the possibility that you are wrong.
If you are wrong, explain clearly what needs to be clarified, improved, or changed in the rubric criteria and guidelines.

In the end, express your final verdict as one of the following three json objects:

```json
{{
    "Instruction Following": "No Issues"
}}
```

```json
{{
    "Instruction Following": "Minor Issue(s)"
}}
```

```json
{{
    "Instruction Following": "Major Issue(s)"
}}
```

<END OF RUBRICS>

# Your task
## User query
<|begin_of_query|>
{user_request}
<|end_of_query|>

**NOTE: CONTEXT DOCUMENT REDACTED FROM USER QUERY. ASSUME THAT IT CONTAINS THE NECESSARY INFORMATION WHICH IS REFERENCED IN THE RESPONSES BELOW.**

## Test Response:
<|begin_of_test_response|>
{response_a}
<|end_of_test_response|>

## Baseline Response:
<|begin_of_baseline_response|>
{response_b}
<|end_of_baseline_response|>

Please write your analysis and final verdict for the test response.""".strip()

    prompt = ineligible_responses_filter_no_context_prompt.format(
        user_request=user_request, response_a=response_a, response_b=response_b
    )
    s = "You are a helpful assistant. Your task is to evaluate the quality of a Response."
    judge = kbench.llms[judge]
    try:
        with kbench.chats.new():
            evaluation_text = judge.prompt([{"role": "system", "content": s}, {"role": "user", "content": prompt}])
    except Exception as ex:
        print(f"Error getting llm response: {ex}")
        evaluation_text = ""
    
    parsed = extract_instruction_json(evaluation_text)

    return 'Major Issue(s)' not in parsed['Instruction Following'], evaluation_text, parsed['Instruction Following']

## Grounding Prompts and Helper Functions

In [6]:
UFG_REV21_PROMPT = """
## Task: Evaluate Factual Accuracy of Model Response Against Provided Context

Your goal is to evaluate whether each sentence in the model's response is factually accurate **solely based on the provided context document**. You must act as a strict fact-checker. Do not use any external knowledge or make assumptions beyond what is stated in the text.

**Output Format:**
For each sentence in the response, output a JSON object with the following fields, representing an evaluation of a single sentence from the model's response:

* `sentence`: (string) The exact sentence from the model's response being evaluated.
* `label`: (string) Your verdict for the sentence. Must be one of:
    * `supported`: The sentence is fully and accurately supported by the provided context document.
    * `not_supported`: The sentence is NOT accurately supported by the provided context document. This includes sentences that are partially correct but contain inaccuracies, unsupported inferences, misattributions, or ignore crucial context.
    * `no_rad`: The sentence is introductory, concluding, conversational filler, or otherwise does not make a factual claim that can be verified against the context (e.g., "Here is the summary:", "Key aspects include:").
* `rationale`: (string) A detailed justification for your `label`.
    * If `supported`, explain *how* the provided excerpt(s) directly and accurately support the *entire* claim made in the sentence. Point out specific phrases or data points.
    * If `not_supported`, clearly explain *why* the sentence is inaccurate. Specify the inaccuracy (e.g., contradicts the text, makes an unsupported inference, misrepresents nuance, misattributes information, ignores contradictory context, uses incorrect terminology). Reference the relevant part(s) of the text or the lack thereof.
    * If `no_rad`, briefly explain why the sentence doesn't require factual verification against the context.
* `excerpt`: (string or null) One or more direct quotes (excerpts) from the context document that are **most relevant** to supporting or refuting the sentence. Concatenate multiple relevant excerpts with a semicolon and space (`; `). If the label is `not_supported` due to a complete lack of information, or if the label is `no_rad`, this field should be `null`.

**Evaluation Criteria - Strict Adherence Required:**

1.  **Explicit Support or Trivial/Strongly-Implied Inferences Only:** A sentence is only `supported` if the information is **explicitly stated** in the context or are logically inferred statements that are strongly implied by the context or trivially implied. Do NOT mark sentences as `supported` if they require inference, assumption, synthesis beyond direct statements, or understanding of implied meaning beyond a very strong or trivial implication. A non-trivial inference that goes well beyond what is strongly or trivially implied by the context is NOT supported.
2.  **Precision and Nuance:** Pay close attention to specific terminology, qualifiers (e.g., "significantly", "most", "some"), quantities, and causal relationships mentioned in the text. Simplifications in the model response are only acceptable if they maintain **full factual accuracy** and do not misrepresent the nuance of the source.
3.  **Completeness and Context:** Evaluate the sentence in the context of the *entire* provided document. A sentence is `not_supported` if it cherry-picks information while ignoring contradictory or significantly qualifying information present elsewhere *in the same document*. The response should not be misleading even if technically based on a fragment of the text.
4.  **Accurate Attribution:** Ensure that claims, descriptions, or characteristics are attributed to the correct subject, object, or scope as defined in the text. Misattributions make a sentence `not_supported`.
5.  **No Hallucinations:** The model response must not introduce information, terms, or concepts not present in the context. Such additions make a sentence `not_supported`.
6.  **Sentence-Level Evaluation:** Evaluate each sentence independently based on these criteria.

Output each JSON object on a new line.

**Example Structure:**

```json
{{"sentence": "The first sentence of the model response.", "label": "supported/not_supported/no_rad", "rationale": "Detailed explanation referencing the criteria and text.", "excerpt": "Relevant quote(s) from context; null if none."}}
{{"sentence": "The second sentence of the model response.", "label": "supported/not_supported/no_rad", "rationale": "Detailed explanation referencing the criteria and text.", "excerpt": "Relevant quote(s) from context; null if none."}}
// ... more sentence evaluations
```

**DO NOT** use RAW quotation marks (") in the values of the sentence, rationale, or excerpt fields in the response. Replace any quotations (") with tick (`)
Example: "The response states, `This is the way it goes`". DO NOT attempt any escaping of the quotations marks - just replace them with ticks.

**Now, please analyze the following context and response:**

**User Query:**
{user_query}

**Context:**
{context}

**Response:**
{response}
""".strip()

def clean_json_content_8(
    text: str, start_field: str, end_field: str | None = None
) -> str:
  r"""Cleans the value of a specified field in a JSON-like string.

  This function can clean a field in the middle of a JSON object or the
  very last field. The output of a model might be a string like:

  {
    "sentence": "The first sentence.",
    "rationale": "The first rationale.",
    "excerpt": "  \\"The first excerpt.\\"  ",
    "label": "supported"
  }

  This function can target a specific field's value (e.g., "excerpt"),
  remove extra quotes and escape characters, and reconstruct the string.

  Args:
    text: The full JSON-like string to clean.
    start_field: The name of the field whose value needs to be cleaned.
    end_field: The name of the field that immediately follows the start_field.
      If start_field is the last field in the object, this should be set to
      None.

  Returns:
    The cleaned string if the specified field is found, otherwise the
    original string.
  """

  # The part of the regex that looks for what comes *after* the value.
  # It's either the next field or the closing brace of the JSON object.
  if end_field:
    # Pattern for a field followed by another field.
    # Captures the comma and the next field key to preserve it.
    trailer_pattern = rf'(,\s*"{re.escape(end_field)}":)'
  else:
    # Pattern for the last field in the object.
    # Captures optional whitespace and the closing brace to preserve it.
    trailer_pattern = r"(\s*})"

  pattern = re.compile(
      rf'"{re.escape(start_field)}":(.+?){trailer_pattern}', re.DOTALL
  )

  def _clean_match(match: re.Match[str]) -> str:
    content = match.group(1)
    trailer = match.group(2)

    content = content.replace('"', "")
    content = content.replace("\\", "")
    content = content.replace("\x02", "")
    content = content.strip()

    cleaned_content = f'"{content}"' if content != "null" else "null"
    return f'"{start_field}":{cleaned_content}{trailer}'

  cleaned_text, num_subs = pattern.subn(_clean_match, text, count=1)
  return cleaned_text if num_subs > 0 else text



def parse_ufg_rev21_verdict(
    answer: str, judge: str, verbose: bool = False
) -> tuple[bool, list[dict[str, str]], bool]:
  """Parses a structured JSON answer into a boolean grounding prediction."""
  if "```json" in answer:
    jsonl_chunks = []
    for a in answer.split("```json")[1:]:
      jsonl_chunks.append(a.split("```")[0])
    answer = "\n".join(jsonl_chunks)
  answer = answer.strip()
  answer = answer.replace("}\n", "}\n@\n@\n")
  answer = answer.replace("} \n", "}\n@\n@\n")
  answer = answer.replace("}  \n", "}\n@\n@\n")
  answer = answer.replace("<ctrl75>", "")
  parsed_answers = []
  for line in answer.split("\n@\n@\n"):
    raw_line = line
    try:
      line = line.replace("\n", " ")
      line = line.replace("\\'", "'")
      line = line.replace("<ctrl75>", "")
      # pylint-enable: g-inconsistent-quotes
      line = line.lstrip(",")
      #  Remove any additional newlines that manifest
      #  as a result of the above replacements.
      line = line.replace("\n", " ")
      line = clean_json_content_8(line, "sentence", "label")
      line = clean_json_content_8(line, "label", "rationale")
      line = clean_json_content_8(line, "rationale", "excerpt")
      line = clean_json_content_8(line, "excerpt")
      parsed = json.loads(line)
      if "label" not in parsed:
        parsed["label"] = "unknown"
      parsed_answers.append(parsed)
    except (json.JSONDecodeError, ValueError, TypeError):
      if verbose:
        print(
            f"error with parsing sentence from {judge} model response to json. The full"
            f" raw line is: {raw_line}"
        )

      if '"not_supported"' in raw_line:
        parsed_answers.append({
            "sentence": "",
            "rationale": "",
            "excerpt": "",
            "label": "not_supported",
        })
  if parsed_answers:
    bool_ans = all(
        parsed_answer["label"] in ("supported", "no_rad", "unknown")
        for parsed_answer in parsed_answers
    )
  else:
    if verbose:
      print(
          f"error with parsing json response from {judge} model. The full answer is:"
          f" {answer}"
      )
    bool_ans = False
  # Only returns unsupported and contradictory sentences to use for rationales.
  filtered_parsed_answers = [
      parsed_answer
      for parsed_answer in parsed_answers
      if parsed_answer["label"] in ["not_supported"]
  ]

  return bool_ans, filtered_parsed_answers

def classify_grounding(user_query, context, response, judge):
    prompt = UFG_REV21_PROMPT.format(user_query=user_query, context=context, response=response)
    s = "You are a helpful assistant. You will be provided with a textual context and a model-generated response. Your task is to analyze the response sentence by sentence and classify each sentence according to its relationship with the provided context."
    judge = kbench.llms[judge]
    try:
        with kbench.chats.new():
            ans = judge.prompt([
                {"role": "system", "content": s},
                {"role": "user", "content": prompt}],)
    except Exception as ex:
        print(f"Error get grader llm response: {ex}")
        ans =""
    bool_ans, parsed_answers = parse_ufg_rev21_verdict(ans, judge, verbose=False)
    return bool_ans, ans

In [7]:
@kbench.task(name="facts_grounding_v2_task")
def facts_grounding_v2_task(llm,full_prompt,user_request,context_document)-> dict:
    try:
        response = llm.prompt(full_prompt)
    except Exception as ex:
        print(f"Error getting llm response: {ex}")
        response=" "    
    
    outputs = {}

    quality_bool = False
    for judge in JUDGES:
        with kbench.chats.new():
            judge_response = kbench.llms[judge].prompt(full_prompt)
        outputs[f"quality-{judge}-response"] = judge_response

        rating_bool, ans, rating = _evaluate_quality_no_context(
            user_request,
            response,
            judge_response,
            judge,
        )
        outputs[f"quality-{judge}-text"] = ans
        outputs[f"quality-{judge}-rating"] = rating
        print(f"Judge {judge} says quality={rating}")

        # If one judge says the quality is good, we break out of the loop since future calls to judges for the same response are not needed.
        if rating_bool:
            quality_bool = True
            break
    outputs["quality"] = quality_bool
    
    if quality_bool:
        grounding_bools = []
        for judge in JUDGES:
            bool_ans, ans = classify_grounding(user_request, context_document, response, judge)
            outputs[f"grounding-{judge}-text"] = ans
            outputs[f"grounding-{judge}-bool"] = bool_ans
            grounding_bools.append(bool_ans)
            print(f"Judge {judge} says grounding={bool_ans}")
        outputs["grounding_score"] = sum(grounding_bools) / len(grounding_bools)
    else:
        outputs["grounding_score"] = 0
    return outputs

In [8]:
columns = ['full_prompt','user_request','context_document']
# Evaluate the task on the dataset
runs = facts_grounding_v2_task.evaluate(
    llm=[kbench.llms['google/gemini-2.5-pro']],
    evaluation_data=examples[columns],
    n_jobs=3,
)

[Parallel(n_jobs=3)]: Using backend ThreadingBackend with 3 concurrent workers.


Judge google/gemini-2.5-flash says quality=Minor Issue(s)


Judge google/gemini-2.5-flash says quality=No Issues


Judge google/gemini-2.5-flash says quality=No Issues


Judge google/gemini-2.5-flash says grounding=False


...

Judge openai/gpt-5-2025-08-07 says grounding=True


[Parallel(n_jobs=3)]: Done   3 out of   5 | elapsed:  3.3min remaining:  2.2min


Judge openai/gpt-5-2025-08-07 says grounding=True


Judge openai/gpt-5-2025-08-07 says grounding=False


[Parallel(n_jobs=3)]: Done   5 out of   5 | elapsed:  4.2min finished


In [9]:
def calculate_accuracy_and_ci(run_results):
    # The input 'run_results' is a list of dictionaries (raw run results).
    scores = []
    for result in run_results:
        if "dictResult" in result:
            # Safely get the 'grounding_score' from inside the 'dictResult' dictionary
            score = result["dictResult"].get("grounding_score", 0.0)
            scores.append(score)
        
    df = pd.DataFrame({"grounding_score": scores})
    
    if len(df) > 0:
        mean_score = float(df.grounding_score.mean())
    else:
        mean_score = 0.0
        
    return mean_score

In [10]:
# Calculate accuracy and confidence intervals
run_files = kbench.kaggle.serialization.get_runs_filenames(runs)

# Aggregate run results
kbench.kaggle.serialization.merge_results_from_runfiles(
    run_files=run_files, aggregate_fn=calculate_accuracy_and_ci, delete_run_files=False
)

'facts_grounding_v2_task-Run_aggregated.run.json'

## View Aggregated Results

In the aggregated runfile, the "results" field will store the mean grounding score for the model.

In [11]:
!cat 'facts_grounding_v2_task-Run_aggregated.run.json'

{
  "taskVersion": {
    "versionNumber": 1,
    "name": "facts_grounding_v2_task",
    "description": "",
    "definition": "@kbench.task(name=\"facts_grounding_v2_task\")\ndef facts_grounding_v2_task(llm,full_prompt,user_request,context_document)-> dict:\n    try:\n        response = llm.prompt(full_prompt)\n    except Exception as ex:\n        print(f\"Error getting llm response: {ex}\")\n        response=\" \"    \n    \n    outputs = {}\n\n    quality_bool = False\n    for judge in JUDGES:\n        with kbench.chats.new():\n            judge_response = kbench.llms[judge].prompt(full_prompt)\n        outputs[f\"quality-{judge}-response\"] = judge_response\n\n        rating_bool, ans, rating = _evaluate_quality_no_context(\n            user_request,\n            response,\n            judge_response,\n            judge,\n        )\n        outputs[f\"quality-{judge}-text\"] = ans\n        outputs[f\"quality-{judge}-rating\"] = rating\n        print(f\"Judge {judge} says quality={rat